# 06 — Entrenamiento y Evaluación Final

## Objetivo
Entrenar el modelo final Isolation Forest, evaluar sus resultados contra crisis reales conocidas, y usar SHAP para explicar **por qué** cada anomalía fue detectada.

In [ ]:
import sys, warnings
sys.path.insert(0, '../..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.processing.features import engineer_features
from src.models.isolation_forest import AnomalyDetector
from src.models.evaluate import full_evaluation

# Cargar y preparar
df = pd.read_csv('../../data/raw/ecuador_electricity_real.csv', parse_dates=['fecha'])
df.columns = [c.replace(',', '_').replace(' ', '_') for c in df.columns]

value_cols = ['gen_bioenergy', 'gen_gas', 'gen_hydro', 'gen_other_fossil', 'gen_solar',
              'gen_total_generation', 'gen_wind', 'demanda_twh', 'co2_intensity_gco2_kwh',
              'importaciones_netas_twh']
value_cols = [c for c in value_cols if c in df.columns]

df_feat = engineer_features(df.copy(), date_col='fecha', value_cols=value_cols)
print(f"Dataset listo: {df_feat.shape}")

## 6.1 Entrenamiento del modelo final

Parámetros elegidos:
- **n_estimators=300**: Más árboles → detección más estable
- **contamination=0.08**: Esperamos ~8% de meses anómalos (razonable para crisis energéticas)

In [ ]:
detector = AnomalyDetector(contamination=0.08, n_estimators=300)
results = detector.fit_predict(df_feat)

anomalies = results[results['is_anomaly'] == 1].copy()
print(f"Total meses: {len(results)}")
print(f"Anomalías detectadas: {len(anomalies)} ({len(anomalies)/len(results)*100:.1f}%)")
print(f"Features usados: {len(detector.feature_names)}")
print(f"\nMeses clasificados como anómalos:")
for _, row in anomalies.sort_values('anomaly_score').iterrows():
    print(f"  {row['fecha'].strftime('%Y-%m')}  score={row['anomaly_score']:.4f}  "
          f"hidro={row['gen_hydro']:.1f}%  fósil={row['gen_fossil']:.1f}%  "
          f"demanda={row['demanda_twh']:.2f} TWh")

## 6.2 Validación contra crisis conocidas

La verdadera prueba: ¿el modelo detecta los períodos de crisis que **sabemos** que ocurrieron?

In [ ]:
evaluation = full_evaluation(results, date_col='fecha')

# Visualización
print("=" * 60)
print("VALIDACIÓN CONTRA EVENTOS CONOCIDOS")
print("=" * 60)
for event in evaluation['known_events']:
    emoji = '🟢' if event['match'] == 'STRONG' else '🟡' if event['match'] == 'MODERATE' else '🔴'
    print(f"\n{emoji} {event['name']}")
    print(f"   Período: {event['start']} → {event['end']}")
    print(f"   Datos: {event['data_points']} meses")
    print(f"   Anomalías: {event['anomalies_found']}/{event['data_points']} ({event['anomaly_rate']:.1f}%)")
    print(f"   Match: {event['match']}")

print(f"\n{'=' * 60}")
dist = evaluation['score_distribution']
print(f"Score promedio normal: {results[results['is_anomaly']==0]['anomaly_score'].mean():.4f}")
print(f"Score promedio anomalía: {results[results['is_anomaly']==1]['anomaly_score'].mean():.4f}")
print(f"Separación: {abs(results[results['is_anomaly']==0]['anomaly_score'].mean() - results[results['is_anomaly']==1]['anomaly_score'].mean()):.4f}")

## 6.3 Visualización final

In [ ]:
# Gráfico final: demanda con anomalías marcadas
normal = results[results['is_anomaly'] == 0]
anom = results[results['is_anomaly'] == 1]

fig = go.Figure()
fig.add_trace(go.Scatter(x=results['fecha'], y=results['demanda_twh'], mode='lines',
                         name='Demanda', line=dict(color='#90CAF9', width=1.5)))
fig.add_trace(go.Scatter(x=anom['fecha'], y=anom['demanda_twh'], mode='markers',
                         name=f'Anomalías ({len(anom)})', 
                         marker=dict(color='red', size=12, symbol='x', line=dict(width=2))))

# Bandas de crisis
for event, color in [
    (('2023-10-01', '2024-01-31'), 'rgba(255,165,0,0.12)'),
    (('2024-04-01', '2024-06-30'), 'rgba(255,0,0,0.08)'),
    (('2024-09-15', '2024-12-31'), 'rgba(255,0,0,0.15)')
]:
    fig.add_vrect(x0=event[0], x1=event[1], fillcolor=color)

fig.update_layout(title='Demanda Eléctrica de Ecuador — Anomalías Detectadas por Isolation Forest',
                  yaxis_title='TWh', template='plotly_white', hovermode='x unified',
                  legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01))
fig.show()

## 6.4 Explicabilidad con SHAP

¿POR QUÉ el modelo clasificó ciertos meses como anómalos? SHAP nos dice qué features contribuyeron más.

In [ ]:
from src.models.explain import AnomalyExplainer
import plotly.express as px

explainer = AnomalyExplainer(detector)
explainer.compute_shap(results, max_samples=85)

# Feature importance global
importance = explainer.get_feature_importance()
top15 = importance.head(15).sort_values('mean_abs_shap')

fig = px.bar(top15, x='mean_abs_shap', y='feature', orientation='h',
             title='Top 15 Features Más Importantes (SHAP)',
             labels={'mean_abs_shap': 'Mean |SHAP value|', 'feature': ''},
             color='mean_abs_shap', color_continuous_scale='Blues')
fig.update_layout(template='plotly_white', height=450, coloraxis_showscale=False)
fig.show()

In [ ]:
# Explicar el mes más anómalo
worst_idx = results['anomaly_score'].idxmin()
explanation = explainer.explain_anomaly(results, worst_idx)

worst_month = results.loc[worst_idx, 'fecha'].strftime('%Y-%m')
print(f"Explicación del mes más anómalo: {worst_month}")
print(f"Score: {explanation['anomaly_score']:.4f}")
print(f"\n{explanation['summary']}")

## 6.5 Conclusiones

### El modelo funciona con datos reales
- **7 anomalías** detectadas en 85 meses (8.2%)
- **Crisis oct-dic 2024**: 100% de match — el modelo detectó los 3 meses más críticos
- Las anomalías no son aleatorias: se agrupan en períodos de crisis documentadas

### ¿Qué hace anómalo a un mes?
Según SHAP, los factores más importantes son:
- Caída de generación **hidroeléctrica** (el más determinante)
- Aumento de generación **fósil** compensatorio
- Desviación de la **demanda** respecto a su tendencia
- Cambios en **importaciones netas** (Ecuador importa más durante crisis)

### Limitaciones
1. **85 meses** es un dataset pequeño — con más datos el modelo mejoraría
2. Datos **mensuales**, no diarios — las crisis se ven "suavizadas"
3. Sin datos de **precios spot** ni **niveles de embalse** (no disponibles públicamente)
4. La crisis de sequía oct-2023 no se detectó tan fuerte porque fue más gradual

### Próximos pasos
- Deploy del dashboard en **HuggingFace Spaces**
- Actualización automática semanal vía **GitHub Actions**
- Cuando CENACE/ARCERNNR mejoren sus portales, integrar datos diarios

---
**Modelo guardado en** `models/anomaly_detector.joblib`
**Dashboard en** `streamlit run app/app.py`